# EDA 001.04 — Character Encodings

**Kaggle Data Cleaning Course — Lesson 4**

Ever seen `UnicodeDecodeError` when loading a CSV file? This notebook covers:

1. **What** are character encodings (ASCII, UTF-8, Latin-1, etc.)
2. How to **detect** the encoding of a file
3. How to **read** files with different encodings
4. How to **convert** between encodings
5. Visualizing encoding issues and their fixes

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import chardet

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1 — What Are Character Encodings?

A **character encoding** maps characters (letters, symbols) to numbers (bytes).

| Encoding | Description | Byte size | Characters |
|----------|-------------|-----------|------------|
| **ASCII** | Original standard (1963) | 1 byte | 128 characters (English only) |
| **Latin-1 (ISO 8859-1)** | Extended ASCII | 1 byte | 256 characters (Western European) |
| **UTF-8** | Universal standard | 1–4 bytes | All Unicode (1M+ characters) |
| **UTF-16** | Fixed-width Unicode | 2–4 bytes | All Unicode |

**UTF-8 is the modern default.** Problems arise when a file is encoded in one format but read as another.

In [ ]:
# Demonstrate: same character, different byte representations
char = "é"  # e-acute
print(f"Character: '{char}'")
print(f"  UTF-8 bytes  : {char.encode('utf-8')}   ({len(char.encode('utf-8'))} bytes)")
print(f"  Latin-1 bytes: {char.encode('latin-1')} ({len(char.encode('latin-1'))} byte)")
print(f"  ASCII bytes  : ", end="")
try:
    print(char.encode('ascii'))
except UnicodeEncodeError as e:
    print(f"ERROR — {e}")

## 2 — The Problem: Loading a File with Wrong Encoding

Our dataset has a Latin-1 encoded version. Let's see what happens when we try to read it as UTF-8.

In [ ]:
# This will FAIL because the file is Latin-1 encoded, not UTF-8
try:
    df_bad = pd.read_csv("data/eda_001_04_projects_latin1.csv", encoding="utf-8")
    print("Loaded successfully (unexpected!)")
except UnicodeDecodeError as e:
    print(f"UnicodeDecodeError!\n  {e}")
    print("\n→ This is the classic error you see when the encoding is wrong.")

## 3 — Detecting File Encoding with `chardet`

The `chardet` library can analyze raw bytes and guess the encoding.

In [ ]:
# Detect encoding of the Latin-1 file
with open("data/eda_001_04_projects_latin1.csv", "rb") as f:
    raw_data = f.read()
    result = chardet.detect(raw_data)

print("chardet detection result:")
print(f"  Encoding  : {result['encoding']}")
print(f"  Confidence: {result['confidence']:.2%}")
print(f"  Language  : {result['language']}")

In [ ]:
# Detect encoding of the UTF-8 file for comparison
with open("data/eda_001_04_projects_utf8.csv", "rb") as f:
    raw_data_utf8 = f.read()
    result_utf8 = chardet.detect(raw_data_utf8)

print("UTF-8 file detection:")
print(f"  Encoding  : {result_utf8['encoding']}")
print(f"  Confidence: {result_utf8['confidence']:.2%}")

## 4 — Reading Files with the Correct Encoding

In [ ]:
# Read the Latin-1 file with the correct encoding
df_latin1 = pd.read_csv("data/eda_001_04_projects_latin1.csv",
                          encoding=result["encoding"])
print(f"Successfully loaded Latin-1 file: {df_latin1.shape}")
df_latin1.head()

In [ ]:
# Read the UTF-8 file normally
df_utf8 = pd.read_csv("data/eda_001_04_projects_utf8.csv", encoding="utf-8")
print(f"Successfully loaded UTF-8 file: {df_utf8.shape}")
df_utf8.head()

In [ ]:
# Verify both files produce identical DataFrames
print(f"DataFrames are identical: {df_latin1.equals(df_utf8)}")

## 5 — Visualizing Encoding: Byte-Level Differences

In [ ]:
# Show how special characters differ at byte level
special_chars = ["é", "ü", "ñ", "ö", "ã", "ç", "ä", "û"]

encoding_data = []
for ch in special_chars:
    utf8_bytes = len(ch.encode("utf-8"))
    latin1_bytes = len(ch.encode("latin-1"))
    encoding_data.append({
        "character": ch,
        "UTF-8 bytes": utf8_bytes,
        "Latin-1 bytes": latin1_bytes,
        "UTF-8 hex": ch.encode("utf-8").hex(),
        "Latin-1 hex": ch.encode("latin-1").hex(),
    })

enc_df = pd.DataFrame(encoding_data)
enc_df

In [ ]:
# Bar chart: byte size comparison
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(special_chars))
width = 0.35

bars1 = ax.bar(x - width/2, enc_df["UTF-8 bytes"], width, label="UTF-8", color="steelblue")
bars2 = ax.bar(x + width/2, enc_df["Latin-1 bytes"], width, label="Latin-1", color="coral")

ax.set_xlabel("Character")
ax.set_ylabel("Bytes")
ax.set_title("Byte Size: UTF-8 vs Latin-1 for Special Characters")
ax.set_xticks(x)
ax.set_xticklabels(special_chars, fontsize=14)
ax.legend()
ax.set_ylim(0, 3)

plt.tight_layout()
plt.show()

## 6 — Converting and Saving with a Specific Encoding

In [ ]:
# Convert Latin-1 file to UTF-8 (the standard approach)
output_path = "data/eda_001_04_projects_converted_to_utf8.csv"
df_latin1.to_csv(output_path, index=False, encoding="utf-8")

# Verify the converted file
with open(output_path, "rb") as f:
    result_check = chardet.detect(f.read())
print(f"Converted file encoding: {result_check['encoding']} (confidence: {result_check['confidence']:.2%})")

# Clean up
import os
os.remove(output_path)
print("Temporary file cleaned up.")

## 7 — Visualizing the Dataset Content

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Budget distribution by category
df_utf8.groupby("category")["budget_eur"].mean().sort_values().plot.barh(
    ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Average Budget by Category")
axes[0].set_xlabel("Budget (EUR)")

# Project count by category
df_utf8["category"].value_counts().plot.bar(
    ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Number of Projects by Category")
axes[1].set_ylabel("Count")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Count non-ASCII characters in each text column
def count_non_ascii(series):
    return series.apply(lambda x: sum(1 for c in str(x) if ord(c) > 127)).sum()

text_cols = ["project_name", "category", "description"]
non_ascii_counts = {col: count_non_ascii(df_utf8[col]) for col in text_cols}

fig, ax = plt.subplots(figsize=(8, 4))
pd.Series(non_ascii_counts).plot.bar(ax=ax, color="mediumpurple", edgecolor="white")
ax.set_title("Non-ASCII Characters per Column")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## 8 — Practical Workflow for Encoding Issues

```
1. Try reading with UTF-8 (the default)
     ↓ UnicodeDecodeError?
2. Use chardet to detect encoding
     ↓
3. Re-read with the detected encoding
     ↓
4. Save as UTF-8 for consistency
```

### Key Takeaways

- **UTF-8** is the universal standard — always save your output as UTF-8
- Use `chardet.detect()` to identify unknown encodings
- `pd.read_csv(encoding=...)` is how you specify encoding in pandas
- `errors='replace'` or `errors='ignore'` are escape hatches (but lose data)
- Non-ASCII characters (accents, special symbols) are the usual culprits